# Follow-the-DROW Training Notebook

Trains all five learnable detectors on the **FROG** dataset (default) and plots
training/validation loss and validation AUC after each run.

| Cell | Model | DirectML | Notes |
|------|-------|----------|-------|
| 3 | DrowDetector | yes | cutout-based |
| 4 | DrSpaamDetector | yes | cutout-based |
| 5 | FullScanCNN | **no** | GRU incompatible with DML |
| 6 | SpaceTimeCNN | yes | fastest DML model |
| 7 | FullScanTransformer | **no** | GRU incompatible with DML |

Adjust `epochs`, `patience`, `lr_schedule`, `subsample` etc. in each cell as needed.

In [ ]:
import sys
import os
from pathlib import Path

# train.py lives in the same directory as this notebook
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'figure.dpi': 110})

from train import train_model, _default_args

print('Imports OK')

In [ ]:
def plot_history(history: dict, model_name: str):
    """Plot training/val loss and val AUC from a train_model() history dict."""
    epochs     = history['epochs']
    auc_epochs = history['auc_epochs']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    stopped = history.get('stopped_epoch', epochs[-1] if epochs else 0)
    title   = model_name
    if stopped < (history.get('epochs', [stopped])[-1] if history['epochs'] else stopped):
        title += f'  (early stop @ epoch {stopped})'
    fig.suptitle(title, fontsize=13, fontweight='bold')

    # --- Loss ---
    ax1.plot(epochs, history['train_loss'], label='train', linewidth=2)
    if history['val_loss']:
        ax1.plot(epochs, history['val_loss'], label='val', linewidth=2, linestyle='--')
    if stopped and stopped < epochs[-1]:
        ax1.axvline(stopped, color='red', linestyle=':', linewidth=1.2, label=f'stop={stopped}')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Loss')
    ax1.legend()
    ax1.grid(alpha=0.3)

    # --- AUC ---
    if auc_epochs:
        ax2.plot(auc_epochs, history['val_auc_agnostic'],
                 label='agnostic', linewidth=2, color='black')
        ax2.plot(auc_epochs, history['val_auc_wc'],
                 label='wc (wheelchair)', linestyle='--')
        ax2.plot(auc_epochs, history['val_auc_wa'],
                 label='wa (walker)',     linestyle='--')
        ax2.plot(auc_epochs, history['val_auc_wp'],
                 label='wp (person)',     linestyle='--')
        ax2.set_ylim(0, 1)
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('AUC')
        ax2.set_title('Validation AUC')
        ax2.legend(fontsize=9)
        ax2.grid(alpha=0.3)
        final_agn = history['val_auc_agnostic'][-1]
        ax2.set_title(f'Validation AUC  (final agnostic={final_agn:.1%})')
    else:
        ax2.text(0.5, 0.5, 'No AUC computed\n(set auc_every > 0)',
                 ha='center', va='center', transform=ax2.transAxes, fontsize=11)
        ax2.set_title('Validation AUC')

    plt.tight_layout()
    plt.show()

print('plot_history() defined')

## DrowDetector

Original DROW WNet3xLF2p.  Cutout-based (per-beam polar windows).  
Compatible with DirectML — expect ~10× speedup over CPU.

In [ ]:
args_drow = _default_args(
    detector     = 'drow',
    dataset      = 'frog',
    epochs       = 30,
    lr           = 1e-3,
    dropout      = 0.5,
    patience     = 10,
    lr_schedule  = 'plateau',
    auc_every    = 5,
    out          = Path('weights_drow_frog.pth'),
)

history_drow = train_model(args_drow)
plot_history(history_drow, 'DrowDetector')

## DrSpaamDetector

DR-SPAAM: BeamNeighborConv + AttnSum.  Cutout-based.  
Compatible with DirectML.

In [ ]:
args_drspaam = _default_args(
    detector     = 'drspaam',
    dataset      = 'frog',
    epochs       = 30,
    lr           = 1e-3,
    dropout      = 0.5,
    patience     = 10,
    lr_schedule  = 'plateau',
    auc_every    = 5,
    out          = Path('weights_drspaam_frog.pth'),
)

history_drspaam = train_model(args_drspaam)
plot_history(history_drspaam, 'DrSpaamDetector')

## FullScanCNNDetector

Dilated 1D CNN over beams + GRU over time.  Full-scan input.  
**Not compatible with DirectML** (GRU is unsupported). `force_cpu=True` ensures
the model always runs on CPU even if DirectML is installed.

Tunable: `backbone_channels`, `hidden`, `dropout`

In [ ]:
args_fscnn = _default_args(
    detector          = 'fullscan_cnn',
    dataset           = 'frog',
    epochs            = 30,
    lr                = 1e-3,
    dropout           = 0.3,
    backbone_channels = 64,
    hidden            = 128,
    patience          = 10,
    lr_schedule       = 'plateau',
    auc_every         = 5,
    force_cpu         = True,   # GRU not supported on DirectML
    out               = Path('weights_fullscan_cnn_frog.pth'),
)

history_fscnn = train_model(args_fscnn)
plot_history(history_fscnn, 'FullScanCNNDetector')

## SpaceTimeCNNDetector

2D convolution over the (N_beams × T) space-time grid.  Full-scan input.  
Compatible with DirectML — fastest among full-scan models on GPU.  
Note: on this GPU, CPU training is actually faster than DirectML for this
small model (~7 ms vs ~10 ms per step).  Set `force_cpu=True` to exploit that.

Tunable: `out_channels`, `dropout`

In [ ]:
args_stcnn = _default_args(
    detector     = 'spacetime_cnn',
    dataset      = 'frog',
    epochs       = 30,
    lr           = 1e-3,
    dropout      = 0.2,
    out_channels = 128,
    patience     = 10,
    lr_schedule  = 'plateau',
    auc_every    = 5,
    force_cpu    = True,   # CPU is faster than DML for this small model
    out          = Path('weights_spacetime_cnn_frog.pth'),
)

history_stcnn = train_model(args_stcnn)
plot_history(history_stcnn, 'SpaceTimeCNNDetector')

## FullScanTransformerDetector

Dilated CNN + global beam self-attention + GRU.  Most expressive full-scan model.  
**Not compatible with DirectML** (GRU is unsupported). `force_cpu=True` is set.

Tunable: `backbone_channels`, `n_heads`, `hidden`, `dropout`

In [ ]:
args_fstransformer = _default_args(
    detector          = 'fullscan_transformer',
    dataset           = 'frog',
    epochs            = 30,
    lr                = 5e-4,
    dropout           = 0.2,
    backbone_channels = 64,
    n_heads           = 8,
    hidden            = 128,
    patience          = 10,
    lr_schedule       = 'plateau',
    auc_every         = 5,
    force_cpu         = True,   # GRU not supported on DirectML
    out               = Path('weights_fullscan_transformer_frog.pth'),
)

history_fstransformer = train_model(args_fstransformer)
plot_history(history_fstransformer, 'FullScanTransformerDetector')

## Summary

Compare final val AUC (agnostic) across all trained models.

In [ ]:
import math

results = [
    ('DrowDetector',              history_drow),
    ('DrSpaamDetector',           history_drspaam),
    ('FullScanCNN',               history_fscnn),
    ('SpaceTimeCNN',              history_stcnn),
    ('FullScanTransformer',       history_fstransformer),
]

print(f"{'Model':<28}  {'Final AUC (agnostic)':>20}  {'Stopped epoch':>14}")
print('-' * 68)
for name, h in results:
    auc = h['val_auc_agnostic'][-1] if h['val_auc_agnostic'] else float('nan')
    ep  = h['stopped_epoch']
    auc_str = f'{auc:.1%}' if not math.isnan(auc) else 'N/A'
    print(f"{name:<28}  {auc_str:>20}  {ep:>14}")